# 第97章 K-Means聚类

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 12 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 朴素贝叶斯  →  **本章任务：** K-Means聚类  →  **下一步：** 主成分分析（PCA）
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：面对一批没有现成标签的样本，比如一组客户的消费记录，你想把它们自然分成几群，却说不出每组该叫什么名字——这正是 K-Means 派上用场的地方。它在没有标准答案的情况下，依据样本之间特征相似与否，把数据归成几个组，为后续的客户画像、商品分群、异常识别等分析给出一个自然的起点。


## 本章目标

学完本章，你将能够：

- **理解**：理解「K-Means聚类」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「K-Means聚类」的关键输出指标。
- **迁移**：能把「K-Means聚类」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：手上一批没有标签的数据，连“该分几组”都没有标准答案——这正是聚类的起点。K-Means 把点分成若干堆，每堆找一个“中心”当队长，谁离队长近就归谁。它简单、快速、可解释，适合“一团一团”的分布；但对初始位置敏感、对长条或弯月形分布束手无策。


- 聚类没有天然正确标签
- K-Means 假设欧氏空间中的近似球形簇（打个比方：把点分成几堆，每堆找个“中心”当队长，谁离队长近就归谁；所以它擅长“一团一团”的分布，遇到长条或弯月形就没那么灵。）
- 初始化会影响局部最优
- 簇编号没有顺序且每次可能置换


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Wine 数据聚类 | `pd.crosstab()`、`.fit_transform()`、`.fit()` | 先标准化，再用多个随机初始化提高稳定性。 | 未标准化直接聚类 |
| 比较簇数 | `rows.append()`、`pd.DataFrame()`、`result.round()`、`.fit()` | 惯性必然随 k 下降，应结合轮廓系数和业务可解释性。 | 把肘部图当作唯一客观答案 |


## 例 1｜Wine 数据聚类

先标准化，再用多个随机初始化提高稳定性。


<!-- math-foundation:chapter-97 -->
### 数学推导｜K-Means 最小化簇内平方距离

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜固定中心，分配样本。** $c_i=\arg\min_k\lVert x_i-\mu_k\rVert_2^2$。

**第 2 步｜固定分配，更新中心。** 对第 $k$ 个簇，平方距离和关于 $\mu_k$ 的最小值在样本均值处：

$$
\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i
$$

**第 3 步｜交替执行。** 分配步和更新步都不会增大目标函数，因此算法会收敛到一个局部最优；不同初始中心可能得到不同结果。

**把上面的关系收束为本章计算式：**

$$
\min_{C_1,\ldots,C_K}\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert_2^2
$$

**符号解释：** $\mu_k$ 是第 $k$ 个簇中心。

**代码对应：** 标准化特征，固定 `random_state`，结合 inertia、轮廓系数和业务可解释性选择 $K$。

**使用边界：** K-Means 偏好球状、大小相近的簇，簇编号本身没有顺序含义。


In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

data = load_wine(as_frame=True)
X, y = data.data, data.target
Xs = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, n_init=20, random_state=86).fit(Xs)
print("silhouette:", round(silhouette_score(Xs, km.labels_), 3))
print("与品种标签ARI（仅诊断）:", round(adjusted_rand_score(y, km.labels_), 3))
print(
    pd.crosstab(km.labels_, y, rownames=["cluster"], colnames=["wine_class"])
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：把上面示例的聚类簇数从 3 改成 4，重新运行 K-Means，观察轮廓系数和聚类结果的变化；如果条件允许，再试着只用两列特征（例如 `alcohol` 和 `malic_acid`）做一次聚类。请写清楚：你改了什么、预期会发生什么、实际观察到什么。

先在下面的练习单元格填写代码，再运行自检核对结果；参考答案在下方（可先隐藏）。


In [ ]:
try:
    # 请在下方填写代码
    # 修改思路：换一个簇数（例如 n_clusters=4），或用两列特征重新聚类，观察轮廓系数变化。
    n_clusters_new = (
        4  # 练习时改成你想观察的簇数；完成后可再试 5、6、7，比较不同设置的差异
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜比较簇数

惯性必然随 k 下降，应结合轮廓系数和业务可解释性。


In [ ]:
rows = []
for k in range(2, 8):
    m = KMeans(n_clusters=k, n_init=20, random_state=86).fit(Xs)
    rows.append([k, m.inertia_, silhouette_score(Xs, m.labels_)])
result = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])
display(result.round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 未标准化直接聚类
- 把肘部图当作唯一客观答案
- 把簇编号解释成高低等级
- 用全部变量聚类后再以同一变量描述簇


## 练习与作业

1. 选择轮廓系数最高的 k
2. 重新训练 K-Means
3. 输出各簇样本量

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 97.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“选择轮廓系数最高的 k”。
2. **独立完成**：不复制示例代码，完成“重新训练 K-Means”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出各簇样本量”，用一两句话说明你修改了什么。

### 97.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 97.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用 K-Means 完成无监督分群，理解标准化、簇数选择、惯性与轮廓系数，并用已知标签仅作事后诊断。


### 你已经掌握

- 训练 KMeans
- 理解质心与簇内平方和
- 标准化不同量纲特征
- 使用 silhouette_score 比较簇数


### 需要注意

- 未标准化直接聚类
- 把肘部图当作唯一客观答案
- 把簇编号解释成高低等级
- 用全部变量聚类后再以同一变量描述簇


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：修改簇数后重新聚类，并对比轮廓系数变化
n_clusters_new = 4
km_new = KMeans(n_clusters=n_clusters_new, n_init=20, random_state=86).fit(Xs)
mysil = silhouette_score(Xs, km_new.labels_)

# 再与 5、6、7 对比，观察轮廓系数随簇数的趋势
for k in range(4, 8):
    m = KMeans(n_clusters=k, n_init=20, random_state=86).fit(Xs)
    print(f"k={k} silhouette={silhouette_score(Xs, m.labels_):.3f}")


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
best_k = int(result.loc[result.silhouette.idxmax(), "k"])
best_km = KMeans(n_clusters=best_k, n_init=20, random_state=86).fit(Xs)
cluster_sizes = pd.Series(best_km.labels_).value_counts().sort_index()
print("best k:", best_k, "sizes:", cluster_sizes.to_dict())
